# [초격차] AI 헬스케어 3기 해커톤: 난임 환자 임신 성공 여부 예측

**평가지표:** ROC-AUC  
**전략:** Phase 1 (XGB×Cat Optuna 30회) → Phase 2 (트리 다양성) → Phase 3 (TabNet/FT-Transformer) → Phase 4 (2단계 스태킹)  
**Data Leakage 원칙:** train에서만 fit, test는 transform only


# 실험 로그

| 실험 | 모델 | Feature Engineering | 파라미터 | Score | 비고 | 리더보드 점수 |
|------|------|---------------------|----------|-------|------|------|
| 4/24(금)|
| exp0 | LightGBM | 기본 feature + 파생변수 110개 | C=1.0, max_iter=100 | 0.7385 | | 리더보드 점수 : 0.5 |
| exp1 | XGBoost | 기본 feature + 파생변수 110개 | lr=0.05, depth=6, subsample=0.8 | 0.7392 | | 리더보드 점수 : 0.5|
| exp2 | Catboost | - | iterations=2000, lr=0.05, depth=6 | 0.7397 | | 리더보드 점수 : 0.7412 |
| exp3 | XGBoost x Catboost 앙상블 | 기본 feature + 파생변수 110개 | iterations=2000, lr=0.05, depth=6  | 0.7399 | A. Simple Average (XGB : Cat = 1:1) |
| exp4 | - |  -  |  - | 0.7399 | B. Weighted Average(OOF AUC가 높은 모델 가중치 부여) |
| exp5 | - |  -  |  - | 0.7399 | C. Stacking (메타 모델 : LR) |
| exp6 | - |  -  |  - | 0.7399 | D. Rank Average(확률 대신 순위로 변환 후 평균) |
| exp7 | XGBoost | 기본 feature + 파생변수 110개 | XGB: depth=5 / Cat: depth=5  | 0.7397 |optuna|
| exp8 | Catboost | - | XGB: depth=5 / Cat: depth=5  | 0.7397 |optuna |
| exp7 | XGBoost x Catboost 앙상블(optuna) |  기본 feature + 파생변수 110개  | XGB: depth=5 / Cat: depth=5 | 0.7399 | A. Simple Average (XGB : Cat = 1:1) |
| exp8 | - |  -  |  - | 0.7399 | B. Weighted Average(OOF AUC가 높은 모델 가중치 부여) |
| exp9 | - |  -  |  - | 0.7399 | C. Stacking (메타 모델 : LR) |
| exp10 | - |  -  |  - | 0.7399 | D. Rank Average(확률 대신 순위로 변환 후 평균) | 리더보드 점수 : 0.74162 |
| 4/25(토) |
| exp11 | LGBM x XGBoost x Catboost 앙상블(optuna) |  기본 feature + 파생변수 110개  | LGBM : max_depth=3 / XGB: max_depth=5 / Cat: depth=6 | 0.7400 | |리더보드 점수 :  0.74140  |
| 4/26(일)|
| exp12 | XGBoost + CatBoost + LightGBM + RandomForest + ExtraTree 앙상블(optuna) |  기본 feature + 파생변수 110개  | LGBM : max_depth=3 / XGB: max_depth=5 / Cat: depth=6 | 0.7398 | Stacking (메타모델 : LR) |
| exp13 | XGBoost + CatBoost + LightGBM 앙상블(파생변수 추가) |  기본 feature + 파생변수 110개 + 신규 파생변수 29  | - | 0.73788 | |
| exp14 | seed 앙상블(파생변수 추가) |  기본 feature + 파생변수 110개 + 신규 파생변수 29  | - | 0.73998 | | 리더보드 점수 : 0.74182 |
| 4/27(월)|
| exp15 | seed 앙상블 + 딥러닝(TabNet) |  기본 feature + 파생변수 110개 + 신규 파생변수 29  | - | 0.73970 | 2단계 스태킹 앙상블 |

## 0. 패키지 설치 및 임포트

In [4]:
! pip install -q catboost lightgbm xgboost optuna pytorch-tabnet
! pip install -q koreanize-matplotlib
! pip install pytorch-tabnet

import warnings, os, time
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import koreanize_matplotlib
except:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
N_SPLITS     = 5
N_OPTUNA     = 30        # Optuna 탐색 횟수
np.random.seed(RANDOM_STATE)

TARGET   = "임신 성공 여부"
ID_COL   = "ID"
DATA_DIR = Path(".")

train = pd.read_csv('/kaggle/input/datasets/symin0729/infertility-patient-data/train.csv', encoding="utf-8-sig")
test  = pd.read_csv('/kaggle/input/datasets/symin0729/infertility-patient-data/test.csv',  encoding="utf-8-sig")
sub = pd.read_csv('/kaggle/input/datasets/symin0729/infertility-patient-data/sample_submission.csv', encoding="utf-8-sig")
train.columns = [c.strip() for c in train.columns]
test.columns  = [c.strip() for c in test.columns]

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"양성 비율: {train[TARGET].mean():.4f}  (불균형: {(train[TARGET]==0).sum()}:{(train[TARGET]==1).sum()})")


Train: (256351, 69) | Test: (90067, 68)
양성 비율: 0.2583  (불균형: 190123:66228)


## 1. 인코딩 맵 & 전처리 상수

In [5]:
AGE_MAP = {"만18-34세":0,"만35-37세":1,"만38-39세":2,
           "만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1}
COUNT_MAP = {"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
DONOR_AGE_MAP = {"만20세 이하":0,"만21-25세":1,"만26-30세":2,
                 "만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}
INDUCTION_MAP = {"알 수 없음":0,"기록되지 않은 시행":1,
                 "생식선 자극 호르몬":2,"세트로타이드 (억제제)":3}
EGG_MAP   = {"본인 제공":0,"기증 제공":1,"알 수 없음":-1}
SPERM_MAP = {"배우자 제공":0,"기증 제공":1,"배우자 및 기증 제공":2,"미할당":-1}
CODE_MAP  = {v:i for i,v in enumerate(
    ["TRCMWS","TRDQAZ","TRJXFG","TRVNRY","TRXQMD","TRYBLT","TRZKPL"])}

COUNT_COLS = ["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수",
              "총 임신 횟수","IVF 임신 횟수","DI 임신 횟수",
              "총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
DROP_COLS  = ["착상 전 유전 검사 사용 여부","PGD 시술 여부","PGS 시술 여부",
              "불임 원인 - 여성 요인","난자 채취 경과일","난자 해동 경과일"]
REASON_CATS  = ["현재 시술용","배아 저장용","난자 저장용","기증용","연구용"]
PROC_TYPES   = ["IVF","ICSI","IUI","ICI","GIFT","FER",
                "BLASTOCYST","AH","Generic DI","IVI"]
print("인코딩 맵 설정 완료")


인코딩 맵 설정 완료


## 2. 전처리 함수 (Leakage-free)

In [6]:
def expand_reason(df):
    for cat in REASON_CATS:
        df[f"이유_{cat}"] = df["배아 생성 주요 이유"].fillna("").str.contains(cat).astype(int)
    return df

def expand_procedure(df):
    filled = df["특정 시술 유형"].fillna("Unknown")
    for pt in PROC_TYPES:
        df[f"시술_{pt}"] = (filled.str.upper()
                            .str.replace(" ","",regex=False)
                            .str.contains(pt.upper()).astype(int))
    return df

def preprocess(df, medians=None, fit=False):
    """fit=True → train에서 중앙값 계산 / fit=False → train 중앙값 적용"""
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    col_years = "임신 시도 또는 마지막 임신 경과 연수"
    if col_years in df.columns:
        df["임신_시도_연수_결측"] = df[col_years].isnull().astype(int)
        df[col_years] = df[col_years].fillna(-1)

    col_thaw = "배아 해동 경과일"
    if col_thaw in df.columns:
        df["배아_해동_결측"] = df[col_thaw].isnull().astype(int)
        df[col_thaw] = df[col_thaw].fillna(0)

    fill_zero = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","총 생성 배아 수",
                 "미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
                 "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수",
                 "해동된 배아 수","해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수",
                 "혼합된 난자 수","파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수",
                 "동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
    for c in fill_zero:
        if c in df.columns: df[c] = df[c].fillna(0)

    if medians is None: medians = {}
    for col in ["난자 혼합 경과일","배아 이식 경과일"]:
        if col not in df.columns: continue
        if fit: medians[col] = df[col].median()
        df[col] = df[col].fillna(medians.get(col, 0))

    cont_cols = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수",
                 "이식된 배아 수","미세주입 배아 이식 수","저장된 배아 수",
                 "미세주입 후 저장된 배아 수","해동된 배아 수","해동 난자 수",
                 "수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
                 "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수","배아 해동 경과일"]
    for c in cont_cols:
        if c not in df.columns: continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if fit: medians[c] = df[c].median() if df[c].notna().any() else 0
        df[c] = df[c].fillna(medians.get(c, 0))

    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).fillna(-1).astype(int)
    for col in COUNT_COLS:
        if col in df.columns: df[col] = df[col].map(COUNT_MAP).fillna(-1).astype(int)
    for col in ["난자 기증자 나이","정자 기증자 나이"]:
        if col in df.columns: df[col] = df[col].map(DONOR_AGE_MAP).fillna(-1).astype(int)
    if "시술 유형" in df.columns:
        df["시술 유형"] = (df["시술 유형"] == "IVF").astype(int)
    if "배란 유도 유형" in df.columns:
        df["배란 유도 유형"] = df["배란 유도 유형"].map(INDUCTION_MAP).fillna(0).astype(int)
    if "난자 출처" in df.columns:
        df["난자 출처"] = df["난자 출처"].map(EGG_MAP).fillna(-1).astype(int)
    if "정자 출처" in df.columns:
        df["정자 출처"] = df["정자 출처"].map(SPERM_MAP).fillna(-1).astype(int)
    if "시술 시기 코드" in df.columns:
        df["시술 시기 코드"] = df["시술 시기 코드"].map(CODE_MAP).fillna(-1).astype(int)

    binary_cols = ["배란 자극 여부","단일 배아 이식 여부","착상 전 유전 진단 사용 여부",
                   "남성 주 불임 원인","남성 부 불임 원인","여성 주 불임 원인","여성 부 불임 원인",
                   "부부 주 불임 원인","부부 부 불임 원인","불명확 불임 원인",
                   "불임 원인 - 난관 질환","불임 원인 - 남성 요인","불임 원인 - 배란 장애",
                   "불임 원인 - 자궁경부 문제","불임 원인 - 자궁내막증",
                   "불임 원인 - 정자 농도","불임 원인 - 정자 면역학적 요인",
                   "불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                   "동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df, medians


## 3. 파생변수 함수 (기존 46개 + 신규 29개)

In [7]:
def feature_engineering(df):
    """기존 파생변수 (문서3 기반, 32개)"""
    df = df.copy()
    for c in ["총 생성 배아 수","이식된 배아 수","저장된 배아 수","수집된 신선 난자 수",
              "미세주입에서 생성된 배아 수","미세주입된 난자 수","해동된 배아 수",
              "미세주입 후 저장된 배아 수","총 임신 횟수","총 시술 횟수",
              "클리닉 내 총 시술 횟수","IVF 시술 횟수","동결 배아 사용 여부",
              "신선 배아 사용 여부","기증 배아 사용 여부","시술 유형","시술 당시 나이"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    cause_cols   = [c for c in df.columns if "불임 원인 -" in c]
    df["불임_원인_합계"] = df[cause_cols].sum(axis=1)
    primary_cols = [c for c in ["남성 주 불임 원인","여성 주 불임 원인","부부 주 불임 원인"] if c in df.columns]
    df["주요_불임_원인_합계"] = df[primary_cols].sum(axis=1)

    df["이식_효율"] = np.where(df["총 생성 배아 수"]>0, df["이식된 배아 수"]/df["총 생성 배아 수"], 0)
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI_비율"] = np.where(df["총 생성 배아 수"]>0, df["미세주입에서 생성된 배아 수"]/df["총 생성 배아 수"], 0)
    df["과거_임신_성공률"] = np.where(df["총 시술 횟수"]>0, df["총 임신 횟수"]/(df["총 시술 횟수"]+1), 0)

    for col in ["동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부"]:
        if col not in df.columns: df[col] = 0
    df["배아_전략"] = df["동결 배아 사용 여부"]*1 + df["신선 배아 사용 여부"]*0 + df["기증 배아 사용 여부"]*2
    df["저장_비율"] = np.where(df["총 생성 배아 수"]>0, df["저장된 배아 수"]/df["총 생성 배아 수"], 0)
    df["기증_사용"] = ((df.get("난자 출처", pd.Series(0,index=df.index))==1)|(df.get("정자 출처", pd.Series(0,index=df.index))==1)).astype(int)

    if "해동된 배아 수" in df.columns:
        df["동결배아_활용률"]   = np.where(df["저장된 배아 수"]>0, df["해동된 배아 수"]/df["저장된 배아 수"], 0)
        df["ICSI동결_이식비율"] = np.where(df["이식된 배아 수"]>0, df["미세주입 후 저장된 배아 수"]/df["이식된 배아 수"], 0)
        df["순수동결_주기"]     = ((df["동결 배아 사용 여부"]==1)&(df["신선 배아 사용 여부"]==0)).astype(int)
        remaining               = (df["저장된 배아 수"]-df["해동된 배아 수"]).clip(lower=0)
        df["동결배아_잉여율"]   = np.where(df["저장된 배아 수"]>0, remaining/df["저장된 배아 수"], 0)
        df["동결배아_총량"]     = df["저장된 배아 수"]+df["해동된 배아 수"]

    if "수집된 신선 난자 수" in df.columns:
        age_num = df["시술 당시 나이"].replace(-1, np.nan)
        egg_cnt = df["수집된 신선 난자 수"]
        df["나이x난자수"]       = (age_num*egg_cnt).fillna(0)
        df["나이보정_난자효율"] = egg_cnt/(age_num.fillna(0)+1)
        df["고령저반응"] = ((age_num>=3)&(egg_cnt<egg_cnt.median())).fillna(False).astype(int)
        df["젊고고수확"] = ((age_num<=1)&(egg_cnt>=egg_cnt.quantile(0.75))).fillna(False).astype(int)

    df["배아_손실수"]       = (df["총 생성 배아 수"]-df["이식된 배아 수"]-df["저장된 배아 수"]).clip(lower=0)
    df["배아_총활용률"]     = np.where(df["총 생성 배아 수"]>0,(df["이식된 배아 수"]+df["저장된 배아 수"])/df["총 생성 배아 수"],0).clip(0,1)
    embryo_max              = max(df["총 생성 배아 수"].max()+1, 9)
    df["배아_풍요도"]       = pd.cut(df["총 생성 배아 수"],bins=[-1,0,3,7,embryo_max],labels=[0,1,2,3]).astype(float).fillna(0).astype(int)
    df["다배아이식_플래그"] = (df["이식된 배아 수"]>=3).astype(int)
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI배아_우위비율"] = np.where(df["총 생성 배아 수"]>0, df["미세주입에서 생성된 배아 수"]/df["총 생성 배아 수"], 0)
    denom = df["이식된 배아 수"]+df["저장된 배아 수"]
    df["이식_집중도"] = np.where(denom>0, df["이식된 배아 수"]/denom, 0)

    if "시술 유형" in df.columns:
        age2 = df["시술 당시 나이"].replace(-1,0)
        df["나이x시술유형"]      = age2*df["시술 유형"]
        df["고령IVF"]            = ((df["시술 당시 나이"]>=3)&(df["시술 유형"]==1)).astype(int)
        df["젊은DI"]             = ((df["시술 당시 나이"]<=1)&(df["시술 유형"]==0)).astype(int)
        df["나이_시술_복합코드"] = df["시술 당시 나이"].clip(lower=0)*10+df["시술 유형"]

    if "총 시술 횟수" in df.columns:
        age3  = df["시술 당시 나이"].replace(-1,0)
        trial = df["총 시술 횟수"].replace(-1,0)
        df["나이x총시술횟수"] = age3*trial
        if "클리닉 내 총 시술 횟수" in df.columns:
            df["나이x클리닉시술횟수"] = age3*df["클리닉 내 총 시술 횟수"].replace(-1,0)
        if "IVF 시술 횟수" in df.columns:
            ivf = df["IVF 시술 횟수"].replace(-1,0)
            df["나이xIVF횟수"]  = age3*ivf
            df["시술부담_지수"] = age3*(trial+ivf)/2
        df["고령반복시술"] = ((df["시술 당시 나이"]>=3)&(trial>=3)).astype(int)
    return df


def add_embryo_quality_features(df):
    """기존 배아 품질 파생변수 (문서4 기반, 14개)"""
    out = df.copy()
    for c in ["총 생성 배아 수","수집된 신선 난자 수","미세주입에서 생성된 배아 수","미세주입된 난자 수","이식된 배아 수"]:
        if c in out.columns: out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
    if "배아 이식 경과일" in out.columns:
        out["배아 이식 경과일"] = pd.to_numeric(out["배아 이식 경과일"], errors="coerce")

    out["수정률"]      = np.where(out["수집된 신선 난자 수"]>0, out["총 생성 배아 수"]/out["수집된 신선 난자 수"], np.nan).clip(0,1)
    out["수정률_결측"] = (out["수집된 신선 난자 수"]==0).astype(int)
    out["수정률"]      = out["수정률"].fillna(0)
    out["수정률_양호"] = (out["수정률"]>=0.5).astype(int)
    out["수정률_구간"] = pd.cut(out["수정률"],bins=[-0.01,0.3,0.5,0.7,1.01],labels=[0,1,2,3]).astype(float)

    out["is_D5"]   = np.where(out["배아 이식 경과일"].isnull(), np.nan, (out["배아 이식 경과일"]>=5).astype(float))
    out["D5_결측"]  = out["배아 이식 경과일"].isnull().astype(int)
    out["is_D5"]   = out["is_D5"].fillna(0)
    out["이식단계"] = np.where(out["배아 이식 경과일"].isnull(),-1,
                               np.where(out["배아 이식 경과일"]<=3,0,
                               np.where(out["배아 이식 경과일"]>=5,1,0)))

    out["ICSI수정률"]      = np.where(out["미세주입된 난자 수"]>0, out["미세주입에서 생성된 배아 수"]/out["미세주입된 난자 수"], np.nan).clip(0,1)
    out["ICSI_미시행"]     = (out["미세주입된 난자 수"]==0).astype(int)
    out["ICSI수정률"]      = out["ICSI수정률"].fillna(0)
    out["ICSI수정률_양호"] = (out["ICSI수정률"]>=0.7).astype(int)
    out["ICSI수정률_구간"] = pd.cut(out["ICSI수정률"],bins=[-0.01,0.5,0.7,0.9,1.01],labels=[0,1,2,3]).astype(float)
    out["이식수_최적"]  = out["이식된 배아 수"].isin([1.0,2.0]).astype(int)
    out["최적이식_D5"] = out["이식수_최적"]*out["is_D5"]
    out["배아질_복합"] = out["수정률"].clip(0,1)*0.4 + out["is_D5"]*0.4 + out["이식수_최적"]*0.2
    return out


def add_new_features(df):
    """★ 신규 파생변수 29개 (행 단위 연산 → Leakage 없음)"""
    out = df.copy()
    for c in ["총 생성 배아 수","이식된 배아 수","저장된 배아 수","수집된 신선 난자 수",
              "저장된 신선 난자 수","혼합된 난자 수","해동된 배아 수","파트너 정자와 혼합된 난자 수",
              "배아 이식 경과일","난자 혼합 경과일","시술 당시 나이","시술 유형",
              "총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수",
              "총 임신 횟수","총 출산 횟수","IVF 임신 횟수","IVF 출산 횟수",
              "DI 임신 횟수","DI 출산 횟수","미세주입된 난자 수","미세주입에서 생성된 배아 수"]:
        if c in out.columns: out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    # 그룹 A. 배아 이식 경과일 관련 (corr 0.208~0.252)
    out["혼합_이식_간격"]   = (out["배아 이식 경과일"]-out["난자 혼합 경과일"]).fillna(0)   # corr=0.252
    out["이식일_D5이상"]    = (out["배아 이식 경과일"]>=5).fillna(False).astype(int)          # corr=0.228
    out["이식일x이식수"]    = out["배아 이식 경과일"].fillna(0)*out["이식된 배아 수"].fillna(0) # corr=0.208
    out["이식일_결측"]      = (out["배아 이식 경과일"]==0).astype(int)                        # corr=-0.243
    out["정확히_D5"]        = (out["배아 이식 경과일"]==5.0).astype(int)                      # corr=0.228
    out["D5_최적이식_복합"] = ((out["배아 이식 경과일"]>=5)&(out["이식된 배아 수"].isin([1.0,2.0]))).fillna(False).astype(int)

    # 그룹 B. 배아/난자 품질 강화
    fert_rate = np.where(out["수집된 신선 난자 수"]>0, out["총 생성 배아 수"]/out["수집된 신선 난자 수"],0).clip(0,1)
    out["생성배아_품질복합"] = out["총 생성 배아 수"]*fert_rate                               # corr=0.126
    out["난자_풍부도"]       = pd.cut(out["수집된 신선 난자 수"],bins=[-1,0,4,8,12,999],labels=[0,1,2,3,4]).astype(float).fillna(0).astype(int)  # corr=0.107
    out["신선난자_저장비율"] = np.where(out["수집된 신선 난자 수"]>0, out["저장된 신선 난자 수"]/out["수집된 신선 난자 수"],0)  # corr=-0.056
    out["파트너정자_활용률"] = np.where(out["혼합된 난자 수"]>0, out["파트너 정자와 혼합된 난자 수"]/out["혼합된 난자 수"],0)
    out["전체_배아_효율"]    = np.where(out["수집된 신선 난자 수"]>0,(out["이식된 배아 수"]+out["저장된 배아 수"])/out["수집된 신선 난자 수"],0).clip(0,5)

    # 그룹 C. 시술 이력 강화
    out["초회시술"]         = (out["총 시술 횟수"]==0).astype(int)                            # corr=0.059
    out["초회IVF"]          = ((out["IVF 시술 횟수"]==0)&(out["총 시술 횟수"]<=1)).astype(int)
    out["클리닉_집중도"]    = np.where(out["총 시술 횟수"]>0, out["클리닉 내 총 시술 횟수"]/out["총 시술 횟수"],1.0).clip(0,1)
    out["IVF_임신_전환율"]  = np.where(out["IVF 시술 횟수"]>0, out["IVF 임신 횟수"]/out["IVF 시술 횟수"],0).clip(0,1)
    out["임신_출산_전환율"] = np.where(out["총 임신 횟수"]>0, out["총 출산 횟수"]/out["총 임신 횟수"],0).clip(0,1)
    out["출산_경험"]        = (out["총 출산 횟수"]>0).astype(int)
    out["IVF_출산_경험"]    = (out["IVF 출산 횟수"]>0).astype(int)

    # 그룹 D. 불임 원인 세분화
    male_cols   = [c for c in ["남성 주 불임 원인","남성 부 불임 원인","불임 원인 - 남성 요인",
                               "불임 원인 - 정자 농도","불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                               "불임 원인 - 정자 면역학적 요인"] if c in out.columns]
    female_cols = [c for c in ["여성 주 불임 원인","여성 부 불임 원인","불임 원인 - 난관 질환",
                               "불임 원인 - 배란 장애","불임 원인 - 자궁내막증","불임 원인 - 자궁경부 문제"] if c in out.columns]
    out["남성불임_복합"]    = out[male_cols].fillna(0).sum(axis=1)                             # corr=0.026
    out["여성불임_복합"]    = out[female_cols].fillna(0).sum(axis=1)
    out["복합_불임_여부"]   = ((out["남성불임_복합"]>0)&(out["여성불임_복합"]>0)).astype(int)
    cause_all = [c for c in out.columns if "불임 원인 -" in c]
    age_num = out["시술 당시 나이"].replace(-1,0)
    out["나이x불임원인수"] = age_num*out[cause_all].fillna(0).sum(axis=1)                      # corr=-0.077

    # 그룹 E. 나이 교호작용 강화
    out["나이xD5"]         = age_num*out["이식일_D5이상"]                                      # corr=0.064
    out["나이x수정률"]     = age_num*fert_rate
    out["나이x클리닉횟수"] = age_num*out["클리닉 내 총 시술 횟수"].replace(-1,0)
    return out


## 4. 전체 전처리 파이프라인 & 데이터 준비

In [8]:
def full_pipeline(df, medians=None, fit=False):
    df = expand_reason(df.copy())
    df = expand_procedure(df)
    df.drop(columns=["배아 생성 주요 이유","특정 시술 유형"], inplace=True, errors="ignore")
    df, medians = preprocess(df, medians=medians, fit=fit)
    df = feature_engineering(df)
    df = add_embryo_quality_features(df)
    df = add_new_features(df)
    return df, medians

X_raw      = train.drop(columns=[ID_COL, TARGET], errors="ignore")
y          = train[TARGET]
X_test_raw = test.drop(columns=[ID_COL], errors="ignore")

# ★ train에서만 fit → test에는 train 통계 적용 (Leakage 방지)
X_pp, train_medians = full_pipeline(X_raw, fit=True)
X_test_pp, _        = full_pipeline(X_test_raw, medians=train_medians, fit=False)

common_cols = [c for c in X_pp.columns if c in X_test_pp.columns]
X_pp        = X_pp[common_cols]
X_test_pp   = X_test_pp[common_cols]

print(f"Train: {X_pp.shape} | Test: {X_test_pp.shape}")
print(f"잔여 결측치(Train): {X_pp.isnull().sum().sum()}")
print(f"총 피처 수: {X_pp.shape[1]}개  (기존 46 + 신규 29)")

X_arr      = X_pp.values.astype(np.float32)
y_arr      = y.values
X_test_arr = X_test_pp.values.astype(np.float32)
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


Train: (256351, 147) | Test: (90067, 147)
잔여 결측치(Train): 0
총 피처 수: 147개  (기존 46 + 신규 29)


## Phase 1 — XGB × CatBoost Optuna 튜닝 (각 30회)

**전략:** OOF 기반 AUC를 목적함수로 Optuna 베이즈 최적화. test 정보는 전혀 사용하지 않음.


In [9]:
# ── Phase 1-A: XGBoost Optuna ──────────────────────────────────────────────
print("=== Phase 1-A: XGBoost Optuna 탐색 ===")

def xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 500, 3000, step=100),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth         = trial.suggest_int("max_depth", 4, 9),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 20),
        gamma             = trial.suggest_float("gamma", 0, 5),
        scale_pos_weight  = trial.suggest_float("scale_pos_weight", 1.5, 4.0),
        tree_method       = "hist",
        eval_metric       = "auc",
        early_stopping_rounds = 50,
        random_state      = RANDOM_STATE,
        n_jobs=-1, verbosity=0,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = xgb.XGBClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=[(X_arr[va],y_arr[va])], verbose=False)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_xgb = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB 최적 AUC: {study_xgb.best_value:.5f}")
print(f"XGB 최적 파라미터: {best_xgb_params}")


=== Phase 1-A: XGBoost Optuna 탐색 ===


  0%|          | 0/30 [00:00<?, ?it/s]

XGB 최적 AUC: 0.73983
XGB 최적 파라미터: {'n_estimators': 1800, 'learning_rate': 0.01681590206217679, 'max_depth': 6, 'subsample': 0.7219206134659216, 'colsample_bytree': 0.6872602657464894, 'reg_alpha': 7.452848201378795, 'reg_lambda': 9.211281496853013, 'min_child_weight': 9, 'gamma': 4.14379616379793, 'scale_pos_weight': 2.064196846705621}


In [10]:
# ── Phase 1-B: CatBoost Optuna ─────────────────────────────────────────────
print("=== Phase 1-B: CatBoost Optuna 탐색 ===")

def cat_objective(trial):
    params = dict(
        iterations        = trial.suggest_int("iterations", 500, 3000, step=100),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth             = trial.suggest_int("depth", 4, 10),
        l2_leaf_reg       = trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        bagging_temperature = trial.suggest_float("bagging_temperature", 0, 2),
        random_strength   = trial.suggest_float("random_strength", 0, 2),
        border_count      = trial.suggest_int("border_count", 32, 255),
        auto_class_weights = "Balanced",
        eval_metric       = "AUC",
        early_stopping_rounds = 50,
        random_seed       = RANDOM_STATE,
        verbose           = 0,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = CatBoostClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=(X_arr[va],y_arr[va]), use_best_model=True)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_cat = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cat.optimize(cat_objective, n_trials=30, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"Cat 최적 AUC: {study_cat.best_value:.5f}")
print(f"Cat 최적 파라미터: {best_cat_params}")


=== Phase 1-B: CatBoost Optuna 탐색 ===


  0%|          | 0/30 [00:00<?, ?it/s]

Cat 최적 AUC: 0.73979
Cat 최적 파라미터: {'iterations': 1200, 'learning_rate': 0.02441661265474018, 'depth': 7, 'l2_leaf_reg': 7.790676508309964, 'bagging_temperature': 1.0157813933009723, 'random_strength': 1.1252522473159237, 'border_count': 91}


In [11]:
# ── Phase 1-C: 최적 파라미터로 OOF + Test 예측 ────────────────────────────
print("=== Phase 1-C: 최적 파라미터 전체 학습 ===")

oof_xgb_p1  = np.zeros(len(X_arr))
pred_xgb_p1 = np.zeros(len(X_test_arr))
oof_cat_p1  = np.zeros(len(X_arr))
pred_cat_p1 = np.zeros(len(X_test_arr))

xgb_final_params = {**best_xgb_params,
    "tree_method":"hist","eval_metric":"auc",
    "early_stopping_rounds":100,"random_state":RANDOM_STATE,"n_jobs":-1,"verbosity":0}
cat_final_params = {**best_cat_params,
    "auto_class_weights":"Balanced","eval_metric":"AUC",
    "early_stopping_rounds":100,"random_seed":RANDOM_STATE,"verbose":0}

for fold,(tr,va) in enumerate(skf.split(X_arr,y_arr),1):
    # XGB
    mx = xgb.XGBClassifier(**xgb_final_params)
    mx.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],verbose=False)
    oof_xgb_p1[va]  = mx.predict_proba(X_arr[va])[:,1]
    pred_xgb_p1    += mx.predict_proba(X_test_arr)[:,1]/N_SPLITS
    # Cat
    mc = CatBoostClassifier(**cat_final_params)
    mc.fit(X_arr[tr],y_arr[tr],eval_set=(X_arr[va],y_arr[va]),use_best_model=True)
    oof_cat_p1[va]  = mc.predict_proba(X_arr[va])[:,1]
    pred_cat_p1    += mc.predict_proba(X_test_arr)[:,1]/N_SPLITS
    print(f"  Fold {fold} | XGB={roc_auc_score(y_arr[va],oof_xgb_p1[va]):.5f}  CAT={roc_auc_score(y_arr[va],oof_cat_p1[va]):.5f}")

print(f"\nPhase1 OOF | XGB={roc_auc_score(y_arr,oof_xgb_p1):.5f}  CAT={roc_auc_score(y_arr,oof_cat_p1):.5f}")

# XGB × Cat 가중치 최적화
def neg_auc_p1(w):
    w = np.array(w); w = w/w.sum()
    return -roc_auc_score(y_arr, w[0]*oof_xgb_p1 + w[1]*oof_cat_p1)
res_p1 = minimize(neg_auc_p1, x0=[0.5,0.5], bounds=[(0,1)]*2, method="L-BFGS-B")
w_p1   = np.array(res_p1.x)/np.array(res_p1.x).sum()
oof_blend_p1  = w_p1[0]*oof_xgb_p1 + w_p1[1]*oof_cat_p1
pred_blend_p1 = w_p1[0]*pred_xgb_p1 + w_p1[1]*pred_cat_p1
print(f"Phase1 XGB×Cat 앙상블 AUC: {roc_auc_score(y_arr,oof_blend_p1):.5f}  (w_xgb={w_p1[0]:.3f}, w_cat={w_p1[1]:.3f})")

# Phase 1 제출 저장
sub_p1 = sub.copy(); sub_p1["probability"] = pred_blend_p1
sub_p1.to_csv("../submission_phase1.csv", index=False)
print("✅ submission_phase1.csv 저장 완료")


=== Phase 1-C: 최적 파라미터 전체 학습 ===
  Fold 1 | XGB=0.73758  CAT=0.73749
  Fold 2 | XGB=0.74249  CAT=0.74298
  Fold 3 | XGB=0.73985  CAT=0.73982
  Fold 4 | XGB=0.73826  CAT=0.73812
  Fold 5 | XGB=0.74107  CAT=0.74062

Phase1 OOF | XGB=0.73984  CAT=0.73980
Phase1 XGB×Cat 앙상블 AUC: 0.74004  (w_xgb=0.500, w_cat=0.500)
✅ submission_phase1.csv 저장 완료


## Phase 2 — 트리 다양성 확장

**전략:** LGB (dart 모드) + XGB (두 버전) + Cat + Seed 앙상블로 다양성 극대화


In [12]:
print("=== Phase 2: 트리 다양성 확장 ===")

# LGB Optuna
def lgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 500, 3000, step=100),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 31, 255),
        max_depth        = trial.suggest_int("max_depth", 4, 12),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        boosting_type    = trial.suggest_categorical("boosting_type", ["gbdt","goss"]),  # ← 원래 ["gbdt","dart","goss"] 였던 부분
        class_weight     = "balanced",
        random_state     = RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],
              callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_lgb = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(lgb_objective, n_trials=30, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB 최적 AUC: {study_lgb.best_value:.5f}")


=== Phase 2: 트리 다양성 확장 ===


  0%|          | 0/30 [00:00<?, ?it/s]

LGB 최적 AUC: 0.73935


In [13]:
# Seed 앙상블 (XGB, Cat, LGB 각 3 seed)
SEEDS = [42, 2024, 777]

oof_models_p2  = {}
pred_models_p2 = {}

model_configs = {
    "XGB_s42" : ("xgb", {**xgb_final_params, "random_state":42}),
    "XGB_s2024": ("xgb", {**xgb_final_params, "random_state":2024}),
    "XGB_s777" : ("xgb", {**xgb_final_params, "random_state":777}),
    "CAT_s42"  : ("cat", {**cat_final_params, "random_seed":42}),
    "CAT_s2024": ("cat", {**cat_final_params, "random_seed":2024}),
    "CAT_s777" : ("cat", {**cat_final_params, "random_seed":777}),
    "LGB_s42"  : ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":42,"n_jobs":-1,"verbose":-1}),
    "LGB_s2024": ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":2024,"n_jobs":-1,"verbose":-1}),
    "LGB_s777" : ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":777,"n_jobs":-1,"verbose":-1}),
}

for name,(mtype,params) in model_configs.items():
    oof  = np.zeros(len(X_arr))
    pred = np.zeros(len(X_test_arr))
    for tr,va in skf.split(X_arr,y_arr):
        if mtype=="xgb":
            m = xgb.XGBClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],verbose=False)
        elif mtype=="cat":
            m = CatBoostClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=(X_arr[va],y_arr[va]),use_best_model=True)
        else:
            m = lgb.LGBMClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],
                  callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        pred   += m.predict_proba(X_test_arr)[:,1]/N_SPLITS
    auc = roc_auc_score(y_arr, oof)
    oof_models_p2[name]  = oof
    pred_models_p2[name] = pred
    print(f"  {name:<12} OOF AUC: {auc:.5f}")

# Phase 2 앙상블 (단순 평균)
oof_blend_p2  = np.mean(list(oof_models_p2.values()), axis=0)
pred_blend_p2 = np.mean(list(pred_models_p2.values()), axis=0)
print(f"\nPhase2 앙상블 OOF AUC: {roc_auc_score(y_arr,oof_blend_p2):.5f}")

sub_p2 = sub.copy(); sub_p2["probability"] = pred_blend_p2
sub_p2.to_csv("submission_phase2.csv", index=False)
print("✅ submission_phase2.csv 저장 완료")


  XGB_s42      OOF AUC: 0.73984
  XGB_s2024    OOF AUC: 0.73983
  XGB_s777     OOF AUC: 0.73978
  CAT_s42      OOF AUC: 0.73980
  CAT_s2024    OOF AUC: 0.73978
  CAT_s777     OOF AUC: 0.73973
  LGB_s42      OOF AUC: 0.73935
  LGB_s2024    OOF AUC: 0.73934
  LGB_s777     OOF AUC: 0.73931

Phase2 앙상블 OOF AUC: 0.73993
✅ submission_phase2.csv 저장 완료


## Phase 3 — 딥러닝 모델

### 3-A. TabNet
Attention 기반 정형 데이터 특화 모델. 각 샘플마다 중요한 피처를 동적으로 선택. 결측치 강건.

### 3-B. FT-Transformer  
각 피처를 토큰으로 임베딩 후 Self-Attention으로 피처 간 교호작용 자동 포착.


In [14]:
# ── Phase 3-A: TabNet ──────────────────────────────────────────────────────
print("=== Phase 3-A: TabNet ===")
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    import torch

    # TabNet용 스케일링 (fold 내부에서만 fit → CV Leakage 없음)
    oof_tabnet  = np.zeros(len(X_arr))
    pred_tabnet = np.zeros(len(X_test_arr))

    tabnet_params = dict(
            n_d=16, n_a=16, # 모델 규모 축소
            n_steps=3,      # 단계 축소 (5 -> 3)
            gamma=1.3, n_independent=2, n_shared=2,
            momentum=0.02, epsilon=1e-15,
            seed=RANDOM_STATE, 
            verbose=1,      # ★ 중요: 1로 바꿔서 진행 상황 확인!
            device_name="cpu", # 요청하신 대로 CPU 설정
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-3, weight_decay=1e-5),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params=dict(step_size=10, gamma=0.9),
        )

    pos_weight = (y_arr==0).sum()/(y_arr==1).sum()

    for fold,(tr,va) in enumerate(skf.split(X_arr,y_arr),1):
        # ★ fold 내부에서만 Scaler fit → CV Leakage 없음
        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_arr[tr])
        X_va_sc = sc.transform(X_arr[va])
        X_te_sc = sc.transform(X_test_arr)

        clf = TabNetClassifier(**tabnet_params)
        clf.fit(
            X_tr_sc, y_arr[tr],
            eval_set=[(X_va_sc, y_arr[va])],
            eval_metric=["auc"],
            max_epochs=50,   # 최대 50회만
            patience=10,     # 10번 성적 안 오르면 중단
            batch_size=4096,
            virtual_batch_size=256, # CPU 부하를 위해 살짝 줄임
            weights=1,
        )
        oof_tabnet[va]  = clf.predict_proba(X_va_sc)[:,1]
        pred_tabnet    += clf.predict_proba(X_te_sc)[:,1]/N_SPLITS
        print(f"  Fold {fold} TabNet OOF AUC: {roc_auc_score(y_arr[va],oof_tabnet[va]):.5f}")

    print(f"TabNet 전체 OOF AUC: {roc_auc_score(y_arr,oof_tabnet):.5f}")
    tabnet_available = True

except ImportError:
    print("pytorch-tabnet 미설치. !pip install pytorch-tabnet 후 재실행")
    oof_tabnet  = np.zeros(len(X_arr))
    pred_tabnet = np.zeros(len(X_test_arr))
    tabnet_available = False


=== Phase 3-A: TabNet ===
epoch 0  | loss: 0.74919 | val_0_auc: 0.59465 |  0:00:11s
epoch 1  | loss: 0.67839 | val_0_auc: 0.65496 |  0:00:22s
epoch 2  | loss: 0.6496  | val_0_auc: 0.68268 |  0:00:33s
epoch 3  | loss: 0.62467 | val_0_auc: 0.69665 |  0:00:44s
epoch 4  | loss: 0.61405 | val_0_auc: 0.7017  |  0:00:55s
epoch 5  | loss: 0.60536 | val_0_auc: 0.70741 |  0:01:06s
epoch 6  | loss: 0.60014 | val_0_auc: 0.71026 |  0:01:17s
epoch 7  | loss: 0.59704 | val_0_auc: 0.71259 |  0:01:28s
epoch 8  | loss: 0.59582 | val_0_auc: 0.71466 |  0:01:39s
epoch 9  | loss: 0.59286 | val_0_auc: 0.71611 |  0:01:50s
epoch 10 | loss: 0.59104 | val_0_auc: 0.71711 |  0:02:01s
epoch 11 | loss: 0.5892  | val_0_auc: 0.71859 |  0:02:12s
epoch 12 | loss: 0.58897 | val_0_auc: 0.71918 |  0:02:23s
epoch 13 | loss: 0.58956 | val_0_auc: 0.71939 |  0:02:34s
epoch 14 | loss: 0.5877  | val_0_auc: 0.72005 |  0:02:45s
epoch 15 | loss: 0.58635 | val_0_auc: 0.72135 |  0:02:56s
epoch 16 | loss: 0.58773 | val_0_auc: 0.72152 

In [15]:
# ── Phase 3-B: FT-Transformer ──────────────────────────────────────────────
print("=== Phase 3-B: FT-Transformer ===")
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    class FTTransformer(nn.Module):
        def __init__(self, n_features, d_model=64, nhead=4, num_layers=3, dropout=0.1):
            super().__init__()
            self.embedding = nn.Linear(n_features, d_model)
            encoder_layer  = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
                dropout=dropout, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.classifier  = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model//2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(d_model//2, 1),
            )
        def forward(self, x):
            x = self.embedding(x).unsqueeze(1)  # (B, 1, d_model)
            x = self.transformer(x)
            x = x.squeeze(1)
            return self.classifier(x).squeeze(-1)

    def train_ft_transformer(X_tr, y_tr, X_va, n_features, epochs=50, lr=1e-3):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model  = FTTransformer(n_features).to(device)

        pos_w  = torch.tensor([(y_tr==0).sum()/(y_tr==1).sum()], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

        X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
        y_tr_t = torch.tensor(y_tr, dtype=torch.float32).to(device)
        X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)

        ds  = TensorDataset(X_tr_t, y_tr_t)
        dl  = DataLoader(ds, batch_size=2048, shuffle=True)

        best_state, best_auc = None, 0
        va_preds_best = None

        for epoch in range(epochs):
            model.train()
            for xb, yb in dl:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                va_logits = model(X_va_t).cpu().numpy()
                va_probs  = 1/(1+np.exp(-va_logits))
                auc_va    = roc_auc_score(y_arr[va_idx_global], va_probs)
                if auc_va > best_auc:
                    best_auc   = auc_va
                    best_state = {k:v.clone() for k,v in model.state_dict().items()}
                    va_preds_best = va_probs

        model.load_state_dict(best_state)
        return model, va_preds_best, best_auc, device

    oof_ftt  = np.zeros(len(X_arr))
    pred_ftt = np.zeros(len(X_test_arr))

    for fold,(tr,va) in enumerate(skf.split(X_arr,y_arr),1):
        va_idx_global = va  # train_ft_transformer 내부에서 참조

        # ★ fold 내부에서 Scaler fit
        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_arr[tr]).astype(np.float32)
        X_va_sc = sc.transform(X_arr[va]).astype(np.float32)
        X_te_sc = sc.transform(X_test_arr).astype(np.float32)

        model, va_preds, best_auc, device = train_ft_transformer(
            X_tr_sc, y_arr[tr], X_va_sc, n_features=X_arr.shape[1])

        oof_ftt[va] = va_preds

        device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.eval()
        with torch.no_grad():
            te_logits = model(torch.tensor(X_te_sc).to(device_)).cpu().numpy()
            pred_ftt += (1/(1+np.exp(-te_logits)))/N_SPLITS

        print(f"  Fold {fold} FT-Transformer Best Val AUC: {best_auc:.5f}")

    print(f"FT-Transformer 전체 OOF AUC: {roc_auc_score(y_arr,oof_ftt):.5f}")
    ftt_available = True

except Exception as e:
    print(f"FT-Transformer 오류: {e}")
    oof_ftt  = np.zeros(len(X_arr))
    pred_ftt = np.zeros(len(X_test_arr))
    ftt_available = False


=== Phase 3-B: FT-Transformer ===
  Fold 1 FT-Transformer Best Val AUC: 0.73596
  Fold 2 FT-Transformer Best Val AUC: 0.74112
  Fold 3 FT-Transformer Best Val AUC: 0.73857
  Fold 4 FT-Transformer Best Val AUC: 0.73603
  Fold 5 FT-Transformer Best Val AUC: 0.73878
FT-Transformer 전체 OOF AUC: 0.73779


## Phase 4 — 2단계 스태킹 앙상블

**Level 0:** Phase 2 모델 9개 + TabNet + FT-Transformer (최대 11개)  
**Level 1 Meta Learner:** LightGBM + Logistic Regression  
**원칙:** Level 0 OOF 예측값만 Level 1 입력으로 사용 → Leakage 없음


In [20]:
print("=== Phase 4: 2단계 스태킹 ===")

# ── Level 0 OOF 수집 ──────────────────────────────────────────────────────
all_oof  = {}
all_pred = {}

# Phase 2 트리 모델 9개
for name in model_configs:
    all_oof[name]  = oof_models_p2[name]
    all_pred[name] = pred_models_p2[name]

# 딥러닝 모델
if tabnet_available and oof_tabnet.sum() != 0:
    all_oof["TabNet"]  = oof_tabnet
    all_pred["TabNet"] = pred_tabnet
    print(f"  TabNet 포함 (OOF AUC: {roc_auc_score(y_arr,oof_tabnet):.5f})")

if ftt_available and oof_ftt.sum() != 0:
    all_oof["FTT"]  = oof_ftt
    all_pred["FTT"] = pred_ftt
    print(f"  FT-Transformer 포함 (OOF AUC: {roc_auc_score(y_arr,oof_ftt):.5f})")

print(f"\nLevel 0 모델 수: {len(all_oof)}개")

# Level 0 피처 행렬 구성
meta_train = np.column_stack(list(all_oof.values()))   # (N_train, n_models)
meta_test  = np.column_stack(list(all_pred.values()))  # (N_test,  n_models)

# ── Level 1-A: Meta LightGBM ──────────────────────────────────────────────
print("\n--- Level 1-A: Meta LightGBM ---")

meta_lgb_params = dict(
    n_estimators=500, learning_rate=0.02, num_leaves=15,
    max_depth=4, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)

oof_meta_lgb  = np.zeros(len(meta_train))
pred_meta_lgb = np.zeros(len(meta_test))

for fold,(tr,va) in enumerate(skf.split(meta_train,y_arr),1):
    m = lgb.LGBMClassifier(**meta_lgb_params)
    m.fit(meta_train[tr], y_arr[tr],
          eval_set=[(meta_train[va],y_arr[va])],
          callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
    oof_meta_lgb[va]  = m.predict_proba(meta_train[va])[:,1]
    pred_meta_lgb    += m.predict_proba(meta_test)[:,1]/N_SPLITS
    print(f"  Fold {fold} Meta-LGB OOF AUC: {roc_auc_score(y_arr[va],oof_meta_lgb[va]):.5f}")

print(f"Meta-LGB 전체 OOF AUC: {roc_auc_score(y_arr,oof_meta_lgb):.5f}")

# ── Level 1-B: Meta Logistic Regression ───────────────────────────────────
print("\n--- Level 1-B: Meta Logistic Regression ---")

oof_meta_lr  = np.zeros(len(meta_train))
pred_meta_lr = np.zeros(len(meta_test))

for fold,(tr,va) in enumerate(skf.split(meta_train,y_arr),1):
    # ★ fold 내부에서 Scaler fit
    sc  = StandardScaler()
    mtr = sc.fit_transform(meta_train[tr])
    mva = sc.transform(meta_train[va])
    mte = sc.transform(meta_test)

    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE,
                            class_weight="balanced")
    lr.fit(mtr, y_arr[tr])
    oof_meta_lr[va]  = lr.predict_proba(mva)[:,1]
    pred_meta_lr    += lr.predict_proba(mte)[:,1]/N_SPLITS

print(f"Meta-LR  전체 OOF AUC: {roc_auc_score(y_arr,oof_meta_lr):.5f}")

# ── 최종 앙상블 ────────────────────────────────────────────────────────────
def neg_auc_final(w):
    w = np.array(w); w = w/w.sum()
    return -roc_auc_score(y_arr, w[0]*oof_meta_lgb + w[1]*oof_meta_lr)

res_f  = minimize(neg_auc_final, x0=[0.7,0.3], bounds=[(0,1)]*2, method="L-BFGS-B")
w_f    = np.array(res_f.x)/np.array(res_f.x).sum()
oof_final  = w_f[0]*oof_meta_lgb + w_f[1]*oof_meta_lr
pred_final = w_f[0]*pred_meta_lgb + w_f[1]*pred_meta_lr

print(f"\n최종 메타 앙상블 OOF AUC: {roc_auc_score(y_arr,oof_final):.5f}")
print(f"가중치: Meta-LGB={w_f[0]:.3f}, Meta-LR={w_f[1]:.3f}")

sub_final = sub.copy()
sub_final["probability"] = pred_final
sub_final.to_csv("submission_final.csv", index=False)
print("\n✅ submission_final.csv 저장 완료")
print(sub_final.head())


=== Phase 4: 2단계 스태킹 ===
  TabNet 포함 (OOF AUC: 0.73044)
  FT-Transformer 포함 (OOF AUC: 0.73779)

Level 0 모델 수: 11개

--- Level 1-A: Meta LightGBM ---
  Fold 1 Meta-LGB OOF AUC: 0.73756
  Fold 2 Meta-LGB OOF AUC: 0.74271
  Fold 3 Meta-LGB OOF AUC: 0.74014
  Fold 4 Meta-LGB OOF AUC: 0.73794
  Fold 5 Meta-LGB OOF AUC: 0.74106
Meta-LGB 전체 OOF AUC: 0.73975

--- Level 1-B: Meta Logistic Regression ---
Meta-LR  전체 OOF AUC: 0.73852

최종 메타 앙상블 OOF AUC: 0.73970
가중치: Meta-LGB=0.700, Meta-LR=0.300

✅ submission_final.csv 저장 완료
           ID  probability
0  TEST_00000     0.014979
1  TEST_00001     0.021010
2  TEST_00002     0.342313
3  TEST_00003     0.229424
4  TEST_00004     0.749839


## 결과 요약 & 비교

In [21]:
print("=" * 55)
print("  Phase별 OOF AUC 비교")
print("=" * 55)
results = {
    "Phase1 XGB×Cat (Optuna)": roc_auc_score(y_arr, oof_blend_p1),
    "Phase2 9-model 앙상블"  : roc_auc_score(y_arr, oof_blend_p2),
    "Phase4 스태킹 최종"     : roc_auc_score(y_arr, oof_final),
}
if tabnet_available: results["Phase3 TabNet"]         = roc_auc_score(y_arr, oof_tabnet)
if ftt_available:    results["Phase3 FT-Transformer"] = roc_auc_score(y_arr, oof_ftt)

for k,v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int((v - 0.80) * 1000)
    print(f"  {k:<32} {v:.5f}  {bar}")
print("=" * 55)

# 최종 제출 파일 확인
import os
for fn in ["submission_phase1.csv","submission_phase2.csv","submission_final3.csv"]:
    if os.path.exists(fn):
        df_ = pd.read_csv(fn)
        print(f"{fn}: {len(df_)}행, prob 범위 [{df_['probability'].min():.4f}, {df_['probability'].max():.4f}]")


  Phase별 OOF AUC 비교
  Phase1 XGB×Cat (Optuna)          0.74004  
  Phase2 9-model 앙상블               0.73993  
  Phase4 스태킹 최종                    0.73970  
  Phase3 FT-Transformer            0.73779  
  Phase3 TabNet                    0.73044  
submission_phase2.csv: 90067행, prob 범위 [0.0012, 0.8737]


In [ ]:
# 피처 중요도 (Phase2 모델 기반)
NEW_FEAT_NAMES = {
    "혼합_이식_간격","이식일_D5이상","이식일x이식수","이식일_결측","정확히_D5","D5_최적이식_복합",
    "생성배아_품질복합","난자_풍부도","신선난자_저장비율","파트너정자_활용률","전체_배아_효율",
    "초회시술","초회IVF","클리닉_집중도","IVF_임신_전환율","임신_출산_전환율","출산_경험","IVF_출산_경험",
    "남성불임_복합","여성불임_복합","복합_불임_여부","나이x불임원인수",
    "나이xD5","나이x수정률","나이x클리닉횟수",
}

# 마지막 LGB 모델로 피처 중요도 추출
last_lgb_params = {**best_lgb_params,"class_weight":"balanced","random_state":RANDOM_STATE,"n_jobs":-1,"verbose":-1}
m_fi = lgb.LGBMClassifier(**last_lgb_params)
m_fi.fit(X_arr, y_arr, callbacks=[lgb.log_evaluation(False)])

feat_imp = pd.DataFrame({"feature":X_pp.columns,"importance":m_fi.feature_importances_}).sort_values("importance",ascending=False)
top30 = feat_imp.head(30)
bar_colors = ["#E05C5C" if f in NEW_FEAT_NAMES else "#5B9BD5" for f in top30["feature"]]

plt.figure(figsize=(10,14))
plt.barh(top30["feature"][::-1], top30["importance"][::-1], color=bar_colors[::-1], alpha=0.88, edgecolor="white")
plt.legend(handles=[
    mpatches.Patch(color="#5B9BD5", label="기존 피처"),
    mpatches.Patch(color="#E05C5C", label="★ 신규 추가 피처"),
], fontsize=10)
plt.title("Feature Importance Top 30 (빨간색 = 신규 피처)", fontsize=13, fontweight="bold")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ feature_importance.png 저장 완료")
